In [1]:
import findspark
findspark.init()
from pyspark import SparkConf, SparkContext
conf = SparkConf().setMaster('local').setAppName('Mi programa')
sc = SparkContext(conf = conf)

In [2]:
sc

<SparkContext master=local appName=Mi programa>

In [3]:
lines = sc.textFile('ejemplopyspark.txt')
lines

ejemplopyspark.txt MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0

In [4]:
lines.count()

366

In [5]:
lines.first()

'Aprendizaje automático'

In [6]:
lines2 = lines.sample(fraction = 0.1, withReplacement = False)

In [7]:
lines2.first()

'Índice'

In [8]:
resultado = lines.filter(lambda line: 'Python' in line)

In [9]:
resultado.count()

3

In [10]:
resultado.take(3)

['Muchos lenguajes de programación pueden usarse para implementar algoritmos de aprendizaje automático. Los más populares para 2015 eran R y Python.6\u200b R es muy usado ante todo en el campo académico, mientras que Python es más popular en la empresa privada.',
 '    scikit-learn: biblioteca en Python que interactúa con NumPy y SciPy',
 '    Raschka, Sebastian (2015). Python Machine Learning, Packt Open Source. ISBN 978-1-78355-513-0']

In [13]:
lines.filter(lambda x: any(i.isdigit() for i in x)).count()

56

In [14]:
numeros = lines.filter(lambda x: any(i.isdigit() for i in x))

In [15]:
numeros.persist()

PythonRDD[9] at RDD at PythonRDD.scala:53

In [16]:
numeros.count()

56

In [17]:
import pandas as pd
df = pd.read_csv('datos_abiertos_covid19.csv', nrows = 6000)

In [18]:
df.head(3)

,FECHA_ACTUALIZACION,ID_REGISTRO,ORIGEN,SECTOR,ENTIDAD_UM,SEXO,ENTIDAD_NAC,ENTIDAD_RES,MUNICIPIO_RES,TIPO_PACIENTE,...,CARDIOVASCULAR,OBESIDAD,RENAL_CRONICA,TABAQUISMO,OTRO_CASO,RESULTADO,MIGRANTE,PAIS_NACIONALIDAD,PAIS_ORIGEN,UCI
0,2020-07-30,004228,2,3,25,2,25,25,6,1,...,2,2,2,1,1,1,99,México,99,97
1,2020-07-30,1e6804,2,4,27,1,27,27,4,1,...,2,2,2,2,99,1,99,México,99,97
2,2020-07-30,105b61,2,4,15,2,15,15,9,1,...,2,1,2,2,99,1,99,México,99,97


In [19]:
from pyspark.sql.types import StringType
from pyspark import SQLContext
sqlContext = SQLContext(sc)

dfspark = sqlContext.read.format('csv').option('header','true').option('interSchema','true').load('datos_abiertos_covid19.csv')

In [20]:
dfspark.show(2)

+-------------------+-----------+------+------+----------+----+-----------+-----------+-------------+-------------+-------------+--------------+----------+--------+--------+----+------------+--------+------------------+--------+----+----+--------+------------+--------+--------------+--------+-------------+----------+---------+---------+--------+-----------------+-----------+---+
|FECHA_ACTUALIZACION|ID_REGISTRO|ORIGEN|SECTOR|ENTIDAD_UM|SEXO|ENTIDAD_NAC|ENTIDAD_RES|MUNICIPIO_RES|TIPO_PACIENTE|FECHA_INGRESO|FECHA_SINTOMAS| FECHA_DEF|INTUBADO|NEUMONIA|EDAD|NACIONALIDAD|EMBARAZO|HABLA_LENGUA_INDIG|DIABETES|EPOC|ASMA|INMUSUPR|HIPERTENSION|OTRA_COM|CARDIOVASCULAR|OBESIDAD|RENAL_CRONICA|TABAQUISMO|OTRO_CASO|RESULTADO|MIGRANTE|PAIS_NACIONALIDAD|PAIS_ORIGEN|UCI|
+-------------------+-----------+------+------+----------+----+-----------+-----------+-------------+-------------+-------------+--------------+----------+--------+--------+----+------------+--------+------------------+--------+----+---

In [21]:
dfspark.head(2)

[Row(FECHA_ACTUALIZACION='2020-07-30', ID_REGISTRO='004228', ORIGEN='2', SECTOR='3', ENTIDAD_UM='25', SEXO='2', ENTIDAD_NAC='25', ENTIDAD_RES='25', MUNICIPIO_RES='006', TIPO_PACIENTE='1', FECHA_INGRESO='2020-06-18', FECHA_SINTOMAS='2020-06-04', FECHA_DEF='9999-99-99', INTUBADO='97', NEUMONIA='2', EDAD='38', NACIONALIDAD='1', EMBARAZO='97', HABLA_LENGUA_INDIG='2', DIABETES='2', EPOC='2', ASMA='2', INMUSUPR='2', HIPERTENSION='2', OTRA_COM='2', CARDIOVASCULAR='2', OBESIDAD='2', RENAL_CRONICA='2', TABAQUISMO='1', OTRO_CASO='1', RESULTADO='1', MIGRANTE='99', PAIS_NACIONALIDAD='México', PAIS_ORIGEN='99', UCI='97'),
 Row(FECHA_ACTUALIZACION='2020-07-30', ID_REGISTRO='1e6804', ORIGEN='2', SECTOR='4', ENTIDAD_UM='27', SEXO='1', ENTIDAD_NAC='27', ENTIDAD_RES='27', MUNICIPIO_RES='004', TIPO_PACIENTE='1', FECHA_INGRESO='2020-04-14', FECHA_SINTOMAS='2020-04-14', FECHA_DEF='9999-99-99', INTUBADO='97', NEUMONIA='2', EDAD='30', NACIONALIDAD='1', EMBARAZO='2', HABLA_LENGUA_INDIG='2', DIABETES='2', EPOC

In [22]:
dfspark.count()

968536

In [23]:
dfspark = dfspark.sample(fraction = 0.01, withReplacement = False)

In [24]:
dfspark.count()

9610

In [25]:
df2 = dfspark.na.drop(subset = ['INTUBADO','EPOC','OBESIDAD','EDAD'])

In [26]:
df2 = df2.filter('INTUBADO is not NULL')

In [27]:
df2.count()

9610

In [28]:
dfspark = dfspark.withColumn('EDAD',dfspark['EDAD'].cast('integer'))

In [29]:
df2.printSchema()

root
 |-- FECHA_ACTUALIZACION: string (nullable = true)
 |-- ID_REGISTRO: string (nullable = true)
 |-- ORIGEN: string (nullable = true)
 |-- SECTOR: string (nullable = true)
 |-- ENTIDAD_UM: string (nullable = true)
 |-- SEXO: string (nullable = true)
 |-- ENTIDAD_NAC: string (nullable = true)
 |-- ENTIDAD_RES: string (nullable = true)
 |-- MUNICIPIO_RES: string (nullable = true)
 |-- TIPO_PACIENTE: string (nullable = true)
 |-- FECHA_INGRESO: string (nullable = true)
 |-- FECHA_SINTOMAS: string (nullable = true)
 |-- FECHA_DEF: string (nullable = true)
 |-- INTUBADO: string (nullable = true)
 |-- NEUMONIA: string (nullable = true)
 |-- EDAD: string (nullable = true)
 |-- NACIONALIDAD: string (nullable = true)
 |-- EMBARAZO: string (nullable = true)
 |-- HABLA_LENGUA_INDIG: string (nullable = true)
 |-- DIABETES: string (nullable = true)
 |-- EPOC: string (nullable = true)
 |-- ASMA: string (nullable = true)
 |-- INMUSUPR: string (nullable = true)
 |-- HIPERTENSION: string (nullable =

In [37]:
import numpy as np
media = np.mean(dfspark.select('EDAD').collect())

In [38]:
media

42.631529656607704

In [33]:
df2.rdd.getNumPartitions()

2